In [ ]:
!pip install datasets

In [ ]:
import pandas as pd
from datasets import load_dataset

# 1. Download the Swahili subset from Hugging Face
afrisenti_data = load_dataset("masakhane/afrisenti", "swa")

# 2. Convert the 'train' split into a Pandas DataFrame for easy cleaning
df_train = afrisenti_data['train'].to_pandas()

# 3. View the first 5 rows to confirm it worked!
df_train.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/213k [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/51.4k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1810 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/453 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/748 [00:00<?, ? examples/s]

,tweet,label
0,Kwani tanesco wanakataga umeme makusudinadhani...,negative
1,cjawahi kuona content yoyote zaidi ya kuwa ana...,negative
2,Bomu lililokuwa limetegwa ndani ya gari likiwa...,negative
3,Kuna video inasambaa mitandaoni jamaa amemfuma...,negative
4,Viwavijeshi wanapita katika hatua kuu 6 za uku...,negative


In [ ]:
# This will show you the total number of rows
print(len(df_train))

# Or this will give you a full summary of the dataset
df_train.info()

1810
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1810 entries, 0 to 1809
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   tweet   1810 non-null   object
 1   label   1810 non-null   object
dtypes: object(2)
memory usage: 28.4+ KB


In [ ]:
import re

def clean_tweet(text):
    # 1. Standardize casing (lowercase everything)
    text = str(text).lower()

    # 2. Strip emojis and special symbols (keeps only alphanumeric and basic punctuation)
    # This regex removes characters outside the standard ASCII/Latin range
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 3. Normalize elongated characters (e.g., "saaaaana" -> "sana")
    # This looks for any character repeated 3 or more times and replaces it with a single character
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 4. Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function to your AfriSenti dataset
df_train['clean_tweet'] = df_train['tweet'].apply(clean_tweet)

# View the results to compare the original vs. cleaned text
df_train[['tweet', 'clean_tweet']].head(10)

,tweet,clean_tweet
0,Kwani tanesco wanakataga umeme makusudinadhani...,kwani tanesco wanakataga umeme makusudinadhani...
1,cjawahi kuona content yoyote zaidi ya kuwa ana...,cjawahi kuona content yoyote zaidi ya kuwa ana...
2,Bomu lililokuwa limetegwa ndani ya gari likiwa...,bomu lililokuwa limetegwa ndani ya gari likiwa...
3,Kuna video inasambaa mitandaoni jamaa amemfuma...,kuna video inasambaa mitandaoni jamaa amemfuma...
4,Viwavijeshi wanapita katika hatua kuu 6 za uku...,viwavijeshi wanapita katika hatua kuu 6 za uku...
5,FamiliaNduguMarafiki wana hofu kubwa juu ya Wa...,familiandugumarafiki wana hofu kubwa juu ya wa...
6,Suala la miundombinu katika maeneo ya uchimbaj...,suala la miundombinu katika maeneo ya uchimbaj...
7,Klabu ya imemfukuza kocha wake sababu ikiwa ni...,klabu ya imemfukuza kocha wake sababu ikiwa ni...
8,Waliomua kinyama mtoto wasakwa gt,waliomua kinyama mtoto wasakwa gt
9,Mwenyekiti wa kijiji cha Kilombero 1 Wilayani ...,mwenyekiti wa kijiji cha kilombero 1 wilayani ...


In [ ]:
import pandas as pd

# 1. Load your new synthetic dataset from INSIDE the sample_data folder
df_synthetic = pd.read_csv('sample_data/kenyan_sheng_swahili_sentiment_dataset.csv')

# 2. Rename columns in df_train (AfriSenti) to match df_synthetic so we can merge them
df_train = df_train.rename(columns={'tweet': 'text', 'label': 'sentiment'})

# 3. Combine both datasets into one master dataframe
df_master = pd.concat([df_train[['text', 'sentiment']], df_synthetic[['text', 'sentiment']]], ignore_index=True)

# 4. Apply your text cleaning pipeline to the master dataset
df_master['clean_text'] = df_master['text'].apply(clean_tweet)

# 5. Check the class balance (Positive, Negative, Neutral)
print(df_master['sentiment'].value_counts())

# View a sample of the final data
df_master.sample(5)

NameError: name 'df_train' is not defined

In [ ]:
import pandas as pd
import re
from datasets import load_dataset

# 1. Define the cleaning function (just in case Colab forgot this too!)
def clean_tweet(text):
    text = str(text).lower()
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. Re-download the AfriSenti Swahili dataset
print("Fetching AfriSenti data from Hugging Face...")
afrisenti_data = load_dataset("masakhane/afrisenti", "swa")
df_train = afrisenti_data['train'].to_pandas()
df_train = df_train.rename(columns={'tweet': 'text', 'label': 'sentiment'})

# 3. Load your synthetic dataset from the sample_data folder
print("Loading synthetic dataset...")
df_synthetic = pd.read_csv('sample_data/kenyan_sheng_swahili_sentiment_dataset.csv')

# 4. Combine both datasets into one master dataframe
print("Merging datasets...")
df_master = pd.concat([df_train[['text', 'sentiment']], df_synthetic[['text', 'sentiment']]], ignore_index=True)

# 5. Apply your text cleaning pipeline
print("Cleaning text...")
df_master['clean_text'] = df_master['text'].apply(clean_tweet)

# 6. Check the final class balance
print("\n--- Final Class Balance ---")
print(df_master['sentiment'].value_counts())

Fetching AfriSenti data from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/213k [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/51.4k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1810 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/453 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/748 [00:00<?, ? examples/s]

Loading synthetic dataset...
Merging datasets...
Cleaning text...

--- Final Class Balance ---
sentiment
Neutral     2500
Positive    2500
Negative    2500
neutral     1072
positive     547
negative     191
Name: count, dtype: int64


In [ ]:
# 1. Unify the labels (capitalize the first letter of everything)
df_master['sentiment'] = df_master['sentiment'].str.capitalize()

# 2. Find the minimum class size (This should be Negative at 2691)
min_class_size = df_master['sentiment'].value_counts().min()

# 3. Downsample each class to match the minimum class size
df_balanced = df_master.groupby('sentiment').sample(n=min_class_size, random_state=42)

# 4. Shuffle the final dataset so the classes are mixed up
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Check the NEW perfect balance
print("\n--- Perfectly Balanced Dataset ---")
print(df_balanced['sentiment'].value_counts())

# Save this to a final CSV so we don't lose it!
df_balanced.to_csv('apt3065_final_training_data.csv', index=False)
print("\nSuccess! Saved to apt3065_final_training_data.csv")


--- Perfectly Balanced Dataset ---
sentiment
Negative    2691
Positive    2691
Neutral     2691
Name: count, dtype: int64

Success! Saved to apt3065_final_training_data.csv


In [ ]:
!pip install transformers torch scikit-learn

import pandas as pd
from transformers import pipeline
from sklearn.metrics import classification_report, f1_score

# 1. Load your perfectly balanced dataset
print("Loading balanced dataset...")
df = pd.read_csv('apt3065_final_training_data.csv')

# 2. Take a small test sample (100 rows) so it runs fast on the free tier
df_test = df.sample(100, random_state=42)

# 3. Load a standard (non-Kenyan) XLM-RoBERTa sentiment model as our baseline
print("Downloading standard XLM-R baseline model (this may take a minute)...")
# We use the cardiffnlp model as it is the standard for multilingual Twitter, but it lacks Sheng!
classifier = pipeline("text-classification", model="cardiffnlp/twitter-xlm-roberta-base-sentiment", max_length=512, truncation=True)

# 4. Run predictions on our Kenyan test set
print("Running baseline predictions...")
def predict_sentiment(text):
    try:
        result = classifier(text)[0]['label']
        # Map model output to our labels
        if result == 'positive': return 'Positive'
        elif result == 'negative': return 'Negative'
        else: return 'Neutral'
    except:
        return 'Neutral'

df_test['baseline_pred'] = df_test['clean_text'].apply(predict_sentiment)

# 5. Calculate how badly it performs (We want a low F1 score here!)
print("\n--- BASELINE EVALUATION RESULTS ---")
print(classification_report(df_test['sentiment'], df_test['baseline_pred']))

# Show a few examples where it failed
failures = df_test[df_test['sentiment'] != df_test['baseline_pred']]
print("\n--- Examples of Model Failures on Sheng/Swahili ---")
print(failures[['clean_text', 'sentiment', 'baseline_pred']].head(5))

Loading balanced dataset...


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Running baseline predictions...

--- BASELINE EVALUATION RESULTS ---
              precision    recall  f1-score   support

    Negative       0.53      0.79      0.64        39
     Neutral       0.67      0.55      0.60        29
    Positive       0.83      0.47      0.60        32

    accuracy                           0.62       100
   macro avg       0.68      0.61      0.61       100
weighted avg       0.67      0.62      0.62       100


--- Examples of Model Failures on Sheng/Swahili ---
                                             clean_text sentiment  \
742   kwanini waelimishaji kwenye ndo walewale inama...   Neutral   
4302  honestly faras customer care walinisaidia wada...  Positive   
1616  nimesubiri bolt dakika kumi bila dere kupatika...  Negative   
4995  endelea kujifunza na kupata taarifa sahihi zai...  Positive   
5982  m-shwari interest ni ok lakini app inalag waka...   Neutral   

     baseline_pred  
742       Negative  
4302      Negative  
1616       Neutral 